In [1]:
import pandas as pd

fraud_df = pd.read_csv("../data/processed/fraud_processed.csv")

print(fraud_df.shape)
print(fraud_df.columns.tolist())

(129146, 19)
['user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id', 'source', 'browser', 'sex', 'age', 'ip_address', 'class', 'ip_int', 'lower_bound_ip_address', 'upper_bound_ip_address', 'country', 'hour_of_day', 'day_of_week', 'time_since_signup', 'transaction_count']


In [3]:
fraud_df.dtypes

user_id                     int64
signup_time                   str
purchase_time                 str
purchase_value              int64
device_id                     str
source                        str
browser                       str
sex                           str
age                         int64
ip_address                float64
class                       int64
ip_int                    float64
lower_bound_ip_address    float64
upper_bound_ip_address    float64
country                       str
hour_of_day                 int64
day_of_week                 int64
time_since_signup         float64
transaction_count           int64
dtype: object

In [2]:
credit_df = pd.read_csv("../data/processed/creditcard_processed.csv")

print(credit_df.shape)
print(credit_df.columns.tolist())

(283726, 31)
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


In [4]:
credit_df.dtypes

Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

In [5]:
# For Fraud Dataset importing necessary libraries
import matplotlib.pyplot as plt

import sys
sys.path.append("..")

from sklearn.model_selection import train_test_split

from src.data_preprocessing import build_preprocessor
from src.imbalance_handler import apply_smote

from src.model_training import (
    train_logistic_regression,
    train_random_forest,
    evaluate_model,
    cross_validate_model
)


### Then load processed Fraud dataset and check it's shape

In [6]:
fraud_df = pd.read_csv(
    "../data/processed/fraud_processed.csv"
)

fraud_df.shape

(129146, 19)

-  Drop unnecessary  columns of fraud_processed dataset, since signup_time, purchase_time, device_id
and ip_address columns that cannot be fed directly into a machine learning model: 

In [7]:
# Drop unnecessary columns:

fraud_df = fraud_df.drop(
    columns=[
        "signup_time",
        "purchase_time",
        "device_id",
        "ip_address"
    ]
)

-  preparing dataset for a machine learning classification task.

In [8]:
X = fraud_df.drop(
    "class",
    axis=1
)

y = fraud_df["class"]

-  Splitting our dataset into training data and testing data so that we can train a model and then evaluate how well it performs on unseen data.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

-  Build the Preprocessor, Learn and Transform Training Data and Transform Test Data

In [10]:
preprocessor = build_preprocessor(
    X_train
)

X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

-  Apply SMOTE on the training dataset

In [11]:
X_train_balanced, y_train_balanced = apply_smote(
    X_train_processed,
    y_train
)

Before SMOTE
class
0    93502
1     9814
Name: count, dtype: int64
After SMOTE
class
0    93502
1    93502
Name: count, dtype: int64


### Interpritation 
### Before SMOTE

| Class          | Count  |
| -------------- | ------ |
| 0 (Legitimate) | 93,502 |
| 1 (Fraud)      | 9,814  |


-  Fraud cases ≈ 9.5% of the dataset
-  Legitimate transactions dominate the dataset
-  A model trained on this data without balancing would likely be biased toward predicting class 0
### After SMOTE 
| Class          | Count  |
| -------------- | ------ |
| 0 (Legitimate) | 93,502 |
| 1 (Fraud)      | 93,502 |
-  Fraud class has been synthetically oversampled to match the majority class
-  The dataset is now fully balanced (1:1 ratio)

#### Train the Logistic Regression model

In [12]:
# Logistic Regression

lr_model = train_logistic_regression(
    X_train_balanced,
    y_train_balanced
)

lr_results = evaluate_model(
    lr_model,
    X_test_processed,
    y_test
)

lr_results

{'AUC_PR': 0.4041865525570814,
 'F1_Score': 0.28500496524329694,
 'Confusion_Matrix': array([[15468,  7908],
        [  732,  1722]])}

#### Logistic regression model result
|                  | Predicted Non-Fraud | Predicted Fraud |
| ---------------- | ------------------- | --------------- |
| Actual Non-Fraud | 15,468              | 7,908           |
| Actual Fraud     | 732                 | 1,722           |

-  Let's see the result based on the standard :
-  Predicted
               0          1

-  Actual 0     TN         FP

-  Actual 1     FN         TP
-  So:

    -  TN = 15468
    -  FP = 7908
    -  FN = 732
    -  TP = 1722
#### What each number means
-  True Negatives (TN)= 15,468

These are legitimate transactions correctly identified as legitimate.
 

-  False Positives (FP)=7,908, These are legitimate transactions incorrectly flagged as fraud. It's Problematic

-  Example:

    -  Customer buys a laptop.
    -  Model says: Fraud.
    -  Reality: Legitimate. This can annoy customers and block valid purchases.

-  False Negatives (FN)= 732, These are fraudulent transactions that the model missed. It's Very costly

-  Example:

    -  Fraud occurs.
    -  Model says: Legitimate.
    -  Reality: Fraud. It leads mmoney may be lost.

-  True Positives (TP)=1,722. Fraud transactions correctly detected.It's Very important

#### Fraud Detection Perspective

-  Total fraud cases:1722 + 732= 2454

-  Detected fraud: 1722

-  Detection rate: 1722 / 2454 ≈ 70.2%

-  So the model catches about:70% of fraud cases. That's not bad,
#### F1 Score Interpretation

-  Result: F1_Score = 0.285

The F1 score balances: Precision and Recall

-  Formula:F1=2⋅ Precision+Recall/ Precision⋅Recall
    -  Calculate Precision: Precision measures: Of all transactions predicted as fraud, how many were actually fraud?
    -  Precision = TP / (TP + FP)
    Precision =1722 / (1722 + 7908) ≈ 0.179

    -  Precision ≈ 17.9% which means When the model says "Fraud",
it is correct only about 18% of the time. Indicates many false alarms.
-  Calculate Recall to measures: Of all actual frauds, how many did we catch?

-  Recall = TP / (TP + FN) = 1722 / (1722 + 732) ≈ 0.702

-  Recall ≈ 70.2% which means The model catches about 70% of all fraud cases.This is fairly good.
#### AUC-PR Interpretation

-  Result: AUC_PR = 0.404. This is the Area Under the Precision-Recall Curve.

-  For fraud detection, AUC-PR is often more useful than accuracy because the classes are imbalanced.

-  What does AUC-PR measure? It evaluates how well the model separates:

    -  Fraud
    vs
    -  Non-Fraud across all classification thresholds.
| AUC-PR    | Interpretation |
| --------- | -------------- |
| < 0.10    | Very poor      |
| 0.10–0.30 | Weak           |
| 0.30–0.50 | Moderate       |
| 0.50–0.70 | Good           |
| > 0.70    | Strong         |
-  Our ur result: 0.404

-  This indicates: Moderate fraud-detection ability.
-  The model is learning meaningful patterns but still has room for improvement.